## The goal is to find revenue growth opportunities
## Based on EDA

In [49]:
import pandas as pd
import sqlite3

In [50]:
conn = sqlite3.connect('../data/database/data_mart.db') 

## Problem 2: High dependence on SP state

---

## Hypothesis 1: Seller concentration in SP drives regional imbalance

In [51]:
revenue_by_states = pd.read_sql(
    """
    SELECT
        customer_state,
        SUM(revenue) / (SELECT SUM(revenue) FROM data_mart) AS percent_of_total_revenue
    FROM data_mart
    GROUP BY customer_state
    ORDER BY percent_of_total_revenue DESC
    LIMIT 5
    """, conn)
revenue_by_states.set_index('customer_state', inplace=True)
revenue_by_states

,percent_of_total_revenue
customer_state,
SP,0.374053
RJ,0.134125
MG,0.117225
RS,0.055815
PR,0.050580


In [ ]:
df_sellers_geo = pd.read_csv('../data/processed/05_olist_sellers_dataset.csv')
df_sellers_geo.groupby('seller_state')['seller_id'].count().sort_values(ascending=False)

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
MS       5
RN       5
MT       4
RO       2
SE       2
AM       1
AC       1
PI       1
MA       1
PA       1
Name: seller_id, dtype: int64

In [53]:
df_sellers_geo.to_sql('sellers_geo',conn,if_exists='replace',index=False)
pd.read_sql(
    """
    SELECT
        AVG(data_mart.revenue),
        sellers_geo.seller_state
    FROM data_mart
    JOIN order_items ON data_mart.order_id = order_items.order_id
    JOIN sellers_geo ON order_items.seller_id = sellers_geo.seller_id
    GROUP BY sellers_geo.seller_state
    HAVING COUNT(DISTINCT data_mart.order_id) > 50
    ORDER BY AVG(data_mart.revenue)
    """,conn)

,AVG(data_mart.revenue),seller_state
0,117.511507,SP
1,118.682694,MA
2,120.406234,DF
3,133.125907,MG
4,143.928537,MT
5,153.527695,ES
6,155.452444,PR
7,156.694932,SC
8,157.995521,GO
9,169.175418,RJ


In [54]:
pd.read_sql(
    """
    SELECT 
        COUNT(DISTINCT order_items.order_id)/COUNT(DISTINCT sellers_geo.seller_id) AS orders_per_seller,
        sellers_geo.seller_state
    FROM sellers_geo
    JOIN order_items ON order_items.seller_id = sellers_geo.seller_id
    GROUP BY sellers_geo.seller_state
    HAVING COUNT(DISTINCT order_items.seller_id) > 50
    ORDER BY orders_per_seller DESC
    """,conn)

,orders_per_seller,seller_state
0,37,SP
1,32,MG
2,25,RJ
3,21,PR
4,19,SC
5,15,RS


In [55]:
pd.read_sql(
    """
    SELECT 
        SUM(data_mart.revenue)/COUNT(DISTINCT data_mart.order_id) AS average_bill,
        sellers_geo.seller_state
    FROM sellers_geo
    JOIN order_items ON order_items.seller_id = sellers_geo.seller_id
    JOIN data_mart ON data_mart.order_id = order_items.order_id
    GROUP BY sellers_geo.seller_state
    HAVING COUNT(DISTINCT order_items.seller_id) > 50
    ORDER BY average_bill DESC
    """,conn)

,average_bill,seller_state
0,281.949909,RS
1,252.350060,RJ
2,241.903697,SC
3,240.398356,PR
4,197.289058,MG
5,190.327218,SP


**Result:**
- about 60% of sellers are located in SP  
- about 40% of orders and revenue come from SP  
- Revenue per seller in SP is among the lowest (117)  
- Orders per seller are highest (37), while AOV is lowest (190)

- Other regions (RS, RJ, SC, PR):
  - Lower orders per seller  
  - Higher AOV (240 – 280)  
  - Higher revenue per seller  

**Conclusion:** Confirmed

---

## Hypothesis 2: Traffic / demand imbalance (not testable)

**Note:**
- No data on marketing channels or ad spend  

**Conclusion:** Not testable

In [56]:
conn.close()